In [ ]:
# Add this as a new cell at the beginning
"""
# 📈 Stock Price Prediction - Advanced Exploratory Data Analysis

This notebook provides comprehensive EDA with interactive visualizations and statistical analysis.

## Table of Contents
1. Data Loading & Validation
2. Statistical Summary
3. Price Analysis
4. Volume Analysis
5. Returns Analysis
6. Technical Indicators
7. Seasonality & Patterns
8. Correlation Analysis
9. Stationarity Tests
10. Advanced Analytics
"""

# Enhanced data loading with validation
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import StockDataLoader
from src.preprocess import StockDataPreprocessor
from src.visualization import StockVisualizer
from utils.config import STOCK_SYMBOL
from utils.helpers import print_section

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
%matplotlib inline

# Initialize loader
loader = StockDataLoader()
df_raw = loader.load_stock_data(STOCK_SYMBOL, force_download=False)

# Validate data
validation = loader.validate_data(df_raw)
print("\n✅ Data Validation Results:")
for key, value in validation.items():
    print(f"  {key:20s}: {value}")

# Enhanced summary statistics with additional metrics
def enhanced_summary(df):
    """Generate enhanced summary statistics"""
    summary = df.describe()
    
    # Add additional metrics
    metrics = {
        'skewness': df.skew(),
        'kurtosis': df.kurtosis(),
        'range': df.max() - df.min(),
        'cv': df.std() / df.mean() * 100,  # Coefficient of variation
        'missing': df.isnull().sum()
    }
    
    return summary, pd.DataFrame(metrics)

summary, metrics = enhanced_summary(df_raw[['Open', 'High', 'Low', 'Close', 'Volume']])
print("\n📊 Enhanced Statistics:")
print(summary)
print("\n📊 Additional Metrics:")
print(metrics)

# Interactive price chart with Plotly
visualizer = StockVisualizer(df_raw)
fig = visualizer.plot_price_history_interactive()
fig.show()

# Volume analysis with moving averages
fig = make_subplots(rows=2, cols=1, subplot_titles=('Volume History', 'Volume Analysis'))
fig.add_trace(
    go.Bar(x=df_raw['Date'], y=df_raw['Volume'], name='Volume'),
    row=1, col=1
)
# Add volume moving averages
for period, color in [(20, 'orange'), (50, 'green')]:
    ma = df_raw['Volume'].rolling(period).mean()
    fig.add_trace(
        go.Scatter(x=df_raw['Date'], y=ma, name=f'MA {period}', line=dict(color=color)),
        row=2, col=1
    )
fig.update_layout(height=600, template='plotly_dark')
fig.show()

# Advanced returns analysis
returns = df_raw['Close'].pct_change().dropna() * 100

fig = make_subplots(rows=2, cols=2, 
                    subplot_titles=('Returns Distribution', 'Q-Q Plot',
                                   'Box Plot', 'Autocorrelation'))

# Distribution
fig.add_trace(
    go.Histogram(x=returns, nbinsx=50, name='Returns'),
    row=1, col=1
)

# Q-Q plot
from scipy import stats
qq = stats.probplot(returns, dist="norm")
fig.add_trace(
    go.Scatter(x=qq[0][0], y=qq[0][1], mode='markers', name='Q-Q'),
    row=1, col=2
)

# Box plot
fig.add_trace(
    go.Box(y=returns, name='Returns'),
    row=2, col=1
)

# Autocorrelation
from statsmodels.tsa.stattools import acf
lags = 20
autocorr = acf(returns, nlags=lags)
fig.add_trace(
    go.Bar(x=list(range(lags+1)), y=autocorr, name='ACF'),
    row=2, col=2
)

fig.update_layout(height=600, template='plotly_dark')
fig.show()

# Stationarity tests
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(series, title):
    """Test stationarity using ADF and KPSS tests"""
    # ADF Test
    adf_result = adfuller(series, autolag='AIC')
    # KPSS Test
    kpss_result = kpss(series, regression='c')
    
    print(f"\n📊 Stationarity Tests for {title}:")
    print("-" * 50)
    print(f"ADF Test: p-value = {adf_result[1]:.4f} -> {'Stationary' if adf_result[1] < 0.05 else 'Non-Stationary'}")
    print(f"KPSS Test: p-value = {kpss_result[1]:.4f} -> {'Stationary' if kpss_result[1] > 0.05 else 'Non-Stationary'}")
    
    return adf_result, kpss_result

test_stationarity(df_raw['Close'], 'Close Price')
test_stationarity(returns, 'Returns')

# Seasonality analysis
def plot_seasonality(df):
    """Plot seasonal patterns"""
    fig = make_subplots(rows=2, cols=2, 
                        subplot_titles=('By Month', 'By Quarter',
                                       'By Day of Week', 'By Year'))
    
    # By month
    monthly = df.groupby(df['Date'].dt.month)['Close'].mean()
    fig.add_trace(go.Bar(x=monthly.index, y=monthly.values, name='Monthly'), row=1, col=1)
    
    # By quarter
    quarterly = df.groupby(df['Date'].dt.quarter)['Close'].mean()
    fig.add_trace(go.Bar(x=quarterly.index, y=quarterly.values, name='Quarterly'), row=1, col=2)
    
    # By day of week
    daily = df.groupby(df['Date'].dt.dayofweek)['Close'].mean()
    fig.add_trace(go.Bar(x=daily.index, y=daily.values, name='Daily'), row=2, col=1)
    
    # By year
    yearly = df.groupby(df['Date'].dt.year)['Close'].mean()
    fig.add_trace(go.Scatter(x=yearly.index, y=yearly.values, mode='lines+markers', name='Yearly'), row=2, col=2)
    
    fig.update_layout(height=600, template='plotly_dark', showlegend=False)
    fig.show()

plot_seasonality(df_raw)

# Advanced pattern detection
def detect_patterns(df):
    """Detect common chart patterns"""
    # Simple pattern detection
    patterns = pd.DataFrame(index=df.index)
    
    # Higher highs, higher lows (uptrend)
    patterns['uptrend'] = ((df['Close'] > df['Close'].shift(1)) & 
                          (df['Close'] > df['Close'].shift(2))).rolling(5).mean() > 0.6
    
    # Lower highs, lower lows (downtrend)
    patterns['downtrend'] = ((df['Close'] < df['Close'].shift(1)) & 
                            (df['Close'] < df['Close'].shift(2))).rolling(5).mean() > 0.6
    
    # Inside bars
    patterns['inside_bar'] = ((df['High'] < df['High'].shift(1)) & 
                              (df['Low'] > df['Low'].shift(1)))
    
    # Outside bars
    patterns['outside_bar'] = ((df['High'] > df['High'].shift(1)) & 
                               (df['Low'] < df['Low'].shift(1)))
    
    # Doji
    patterns['doji'] = abs(df['Close'] - df['Open']) / (df['High'] - df['Low']) < 0.1
    
    print("\n🎯 Pattern Detection Summary:")
    for col in patterns.columns:
        count = patterns[col].sum()
        pct = count / len(patterns) * 100
        print(f"  {col:15s}: {count:5d} ({pct:.1f}%)")
    
    return patterns

patterns = detect_patterns(df_raw)

print("\n✅ Enhanced EDA Complete!")